## KMS — envelope encryption & the key hierarchy

**KMS** creates and protects cryptographic keys. Keys live inside FIPS 140-2 validated **hardware security modules** (HSMs) — they never leave KMS in plaintext, and every operation happens inside the HSM. Every call is logged to CloudTrail, so encryption activity is fully audited.

### Key types by ownership

| Type | Created by | Controlled by | Cost | Rotation |
|---|---|---|---|---|
| **AWS managed key** | AWS, automatically | AWS | free | yearly, automatic |
| **Customer managed key (CMK)** | you | you | $1/mo + API calls | optional (yearly or on-demand) |
| **AWS owned key** | AWS, shared | AWS (invisible) | free | AWS handles |

AWS managed keys (`aws/s3`, `aws/ebs`, `aws/rds`) spin up automatically the first time you enable encryption on a service. Reach for a **CMK** when you need control over the key policy, rotation schedule, or cross-account access.

### Envelope encryption

KMS has a hard limit of **4 KB** per direct `Encrypt` call, and a per-region rate limit that bulk encryption would blow through. The pattern AWS uses everywhere to get around this is **envelope encryption**:

1. Call `GenerateDataKey(KeyId)`. KMS returns a **plaintext data key** (DEK) *and* the same DEK **encrypted under your KMS key**.
2. Encrypt the actual data locally with the plaintext DEK — a fast symmetric cipher, no KMS round-trip per byte.
3. Store the encrypted data **beside the encrypted DEK**, and discard the plaintext DEK from memory.
4. To decrypt: send the encrypted DEK to KMS, get the plaintext DEK back, decrypt locally.

The wins stack: large payloads encrypt at local CPU speed, the encrypted DEK is safe to keep next to the data (useless without KMS access), and KMS only ever sees small keys.